# 035_full_load
Runs a Silver script (one plain `SELECT` in `030_scripts/`) and replaces the Silver table with
its result. The script defines the shape; this notebook owns the load pattern.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## Loader

In [ ]:
def silver_full_load(config, batch_id):
    df = with_lineage(run_script(config), batch_id)
    return write_table(df, "silver", config["destination_schema"], config["destination_table"])


LOADERS["035_full_load"] = silver_full_load

## Run on its own

In [ ]:
# Parameters: set a control_id and run all to load one flow on its own (handy for testing).
# Left at 0, this notebook only defines its loader for the orchestrator.
control_id = 0
batch_id = ""

In [ ]:
if control_id:
    print(execute_flow(control_id, batch_id or new_batch_id(), silver_full_load, triggered_by="manual"))